# Google AI Pro × Colab A100 × Strata：125B MoEを動かしてみる

**2026年10月10日の実測環境：** Colab A100-SXM4 40GB、システムRAM約83 GiB、CUDA 13（ドライバ580.82.07）、[Strata](https://github.com/Niko1221/Strata)、**Qwen3.8-Flash-Next（125B MoE）IQ2_XS（約2bit量子化）**、32Kコンテキスト、画像入力オフ。

Google AI ProにはColabコンピューティングユニットが含まれますが、A100の割り当ては保証されません。まず、Colabの **［ランタイム］→［ランタイムのタイプを変更］→［A100 GPU］** を選択し、**高RAM** の設定が選べるなら使用してください。試験環境と同等のRAM・ディスク容量がない場合はセットアップできないことがあります。

**必要容量・時間：** 初回はモデル約68GBのダウンロード、生成される中間データと一時ファイル、必要ならCUDAエンジンのコンパイルに時間がかかります。ディスクは目安として120GB以上の空きがあるか確認してください（必要量はStrataのバージョンで変化）。高性能GPUの利用中はCompute Unitsが消費されます。Colabは常設サーバーではなく、**Colab内での個人的な動作・速度検証**のために使ってください。

**実測例（単発・異条件の測定）：** 長文6,886入力トークンでPrefill **2,572 tok/s**、最大64トークン出力時のDecode **116.9 tok/s**。別の1,024トークン出力テストではDecode **68.0 tok/s**。**116.9 tok/sは1024トークン連続生成の速度ではありません。**

参考：[Strata公式](https://github.com/Niko1221/Strata) / [GoogleのColab特典発表](https://developers.googleblog.com/colab-is-now-part-of-your-google-ai-plan/) / [Colab FAQ](https://research.google.com/colaboratory/faq.html?hl=ja)

## 1. GPU・RAM・SSDを確認し、Strataを取得

`8080`を優先します。すでに他のサービスが使用中なら、自動的に`8081`または`8082`を選びます。成功した環境では、`STRATA_UV=1`でPython環境を作るのがポイントです。

In [ ]:
import os, sys, socket, subprocess
from pathlib import Path

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["free", "-h"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
driver = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], text=True).strip().splitlines()[0]
assert "A100" in gpu, f"このノートブックの検証対象はA100です。実際の割り当て: {gpu}"
assert int(driver.split(".")[0]) >= 580, f"CUDA 13にはNVIDIAドライバ580以上が必要です。現在: {driver}"

def port_is_free(p):
    with socket.socket() as sock:
        try:
            sock.bind(("127.0.0.1", p))
            return True
        except OSError:
            return False

PORT = next((p for p in (8080, 8081, 8082) if port_is_free(p)), None)
assert PORT is not None, "8080〜8082番ポートがすべて使用中です"
os.environ["STRATA_PORT"] = str(PORT)
print("Strataに使用するポート:", PORT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
root = Path("/content/Strata")
if not (root / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Niko1221/Strata.git", str(root)], check=True)

print("取得完了。次のセルでセットアップを実行してください。")

## 2. Strataと量子化モデルをインストール

今回のA100 40GB／83GB RAMに合わせて、**IQ2_XS（約2bit）**、**32,768コンテキスト**、CUDA 13を使います。Colab標準Pythonと`venv`の相性問題を避けるため`STRATA_UV=1`を設定します。

**初回は長時間かかります。** 68GB程度のモデルダウンロード、必要に応じたCUDAコンパイル、モデルの準備があります。下のセルは、元の`subprocess.run()`の代わりにColabの`!`コマンドで実行し、進捗ログを表示します。`--no-start`はセットアップ後にサーバーを起動しないという意味です。途中で中断した場合も、ランタイムを削除せず同じセルを再実行すれば、通常はダウンロードを再開できます。

In [ ]:
import os
from pathlib import Path

os.environ["STRATA_UV"] = "1"
os.environ["PATH"] = (
    "/usr/local/bin:" + os.path.expanduser("~/.local/bin")
    + ":" + os.environ.get("PATH", "")
)
print(f"Strataセットアップ開始（ポート {PORT}）。初回は時間がかかります。", flush=True)

# Colabの ! コマンドで実行すると、ダウンロード・コンパイルのログが見やすくなります。
!cd /content/Strata && bash setup.sh \
  --setup --yes \
  --backend cuda --cuda 13 --gpu 0 \
  --family qwen --model IQ2_XS \
  --context 32768 --vision no \
  --data-dir /content/Strata-data \
  --host 127.0.0.1 --port {PORT} \
  --no-browser --no-start

assert Path("/content/Strata/run-iq2_xs.sh").is_file(), (
    "セットアップが完了していません。上のログの末尾を確認してください。"
)
print("セットアップの起動スクリプトを確認しました。次のセルでサーバーを起動します。")

## 3. サーバーを起動し、モデルの読み込みを待つ

最初の起動では数十GBの重みをRAM・VRAMに配置します。**約2分以上かかることがあります。** `loaded: True`が表示されるまで待ってください。

In [ ]:
import requests, time, subprocess
from pathlib import Path

BASE = f"http://127.0.0.1:{PORT}"
s = requests.Session()
s.trust_env = False

def get_health():
    try:
        r = s.get(BASE + "/health", timeout=3)
        if r.ok:
            j = r.json()
            if j.get("service") == "strata":
                return j
    except (requests.RequestException, ValueError):
        pass
    return None

health = get_health()
if not (health and health.get("loaded")):
    script = Path("/content/Strata/run-iq2_xs.sh")
    assert script.exists(), "セットアップが未完了です（run-iq2_xs.shがありません）"
    log_path = Path(f"/content/strata-{PORT}.log")
    with log_path.open("a") as log:
        proc = subprocess.Popen(["bash", str(script)], cwd="/content/Strata",
                                stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    print(f"サーバー起動 PID={proc.pid}。ロード中です…")
    for i in range(120):  # 最大10分
        health = get_health()
        if health and health.get("loaded"):
            break
        if proc.poll() is not None:
            print(log_path.read_text(errors="replace")[-5000:])
            raise RuntimeError(f"Strataが終了しました（exit {proc.returncode}）")
        if i % 6 == 0:
            print(f"{i * 5} 秒経過。モデルのロード待機中…")
        time.sleep(5)
    else:
        print(log_path.read_text(errors="replace")[-5000:])
        raise TimeoutError("10分待ってもStrataが起動しませんでした。ログを確認してください。")

print("READY:", health)
print("OpenAI互換API:", BASE + "/v1")

## 4. 文章をストリーミング生成＆速度表示

エンジンが返すPrefill・Decodeの内部測定値を最後に表示します。初回と短文での速度は変動するため、必要なら2回以上実行してください。

In [ ]:
import requests, time, json

payload = {
    "model": "qwen3.8-flash-next-iq2_xs",
    "messages": [{"role": "user", "content":
                  "AIが化学メーカーの研究開発をどう変えるか、具体例を交えて日本語で解説してください。"}],
    "max_tokens": 512,
    "reasoning_effort": "none",
    "temperature": 0,
    "stream": True,
}

start = time.perf_counter()
first_text = None
usage, timings = {}, {}
print("=== 文章生成（ストリーミング）===\n")
with s.post(BASE + "/v1/chat/completions", json=payload, stream=True,
            timeout=(10, 900)) as r:
    if not r.ok:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:800]}")
    for line in r.iter_lines(chunk_size=1):
        if not line or not line.startswith(b"data:"):
            continue
        raw = line[5:].strip()
        if raw == b"[DONE]":
            break
        event = json.loads(raw)
        if event.get("error"):
            raise RuntimeError(str(event["error"]))
        for choice in event.get("choices", []):
            piece = (choice.get("delta") or {}).get("content") or ""
            if piece:
                if first_text is None:
                    first_text = time.perf_counter()
                print(piece, end="", flush=True)
        usage = event.get("usage") or usage
        timings = event.get("timings") or timings

elapsed = time.perf_counter() - start
print("\n\n=== ベンチマーク ===")
print("入力トークン:", usage.get("prompt_tokens"))
print("出力トークン:", usage.get("completion_tokens"))
print("Prefill:", timings.get("prompt_per_second"), "tok/s")
print("Decode:", timings.get("predicted_per_second"), "tok/s")
print("総時間:", round(elapsed, 2), "秒")
if first_text:
    print("初回表示まで:", round(first_text - start, 2), "秒")
if timings.get("draft_n"):
    accept = timings.get("draft_n_accepted", 0)
    print("投機的デコード採用率:", round(100 * accept / timings["draft_n"], 1), "%")

## 5. ［任意］長文のPrefill性能を比較する

以下は人工的な繰り返し文章を使った**3条件の単発測定**です。実際の文書処理の速度とは異なる場合があります。出力は最大64トークンのため、ここでのDecode速度を長文生成の値として扱わないでください。

In [ ]:
import uuid, time

paragraph = (
    "Artificial intelligence is transforming scientific research. "
    "Researchers use machine learning to analyze experimental data, "
    "discover new materials, optimize chemical processes, and predict "
    "physical properties. Reliable evaluation requires reproducible "
    "experiments, careful validation, and quantitative comparisons. "
)

for label, chars in [("短文", 100), ("中程度", 20000), ("長文", 50000)]:
    doc = (paragraph * (chars // len(paragraph) + 1))[:chars]
    text = f"Test ID: {uuid.uuid4().hex}\n次の文章を日本語で簡潔に要約してください。\n{doc}"
    data = {
        "model": "qwen3.8-flash-next-iq2_xs", "messages": [{"role": "user", "content": text}],
        "max_tokens": 64, "reasoning_effort": "none", "temperature": 0, "stream": False
    }
    t0 = time.perf_counter()
    r = s.post(BASE + "/v1/chat/completions", json=data, timeout=900)
    r.raise_for_status()
    out = r.json()
    tm, usage = out.get("timings", {}), out.get("usage", {})
    print(f"{label}: 入力{usage.get('prompt_tokens')} tokens | "
          f"Prefill {tm.get('prompt_per_second')} tok/s | "
          f"Decode {tm.get('predicted_per_second')} tok/s | "
          f"キャッシュ再利用 {tm.get('cache_n')} | "
          f"総時間 {time.perf_counter() - t0:.2f} 秒")

## 6. ［任意］Google Driveにモデルをバックアップ

初回セットアップとダウンロードがすべて終わってから実行してください。**モデルの配布元だけでなく、生成されたモデル用データも含めて**コピーするため、Drive側の必要容量はモデルのダウンロードサイズ（約68GB）より大きくなる可能性があります。次回ランタイムではこのデータを復元した後、Strata本体のインストールを行ってください。Drive上から直接推論せず、Colab側のSSDに戻す方が安定しやすいです。

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount("/content/drive")
backup = Path("/content/drive/MyDrive/Strata-backup")
backup.mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", "--info=progress2",
                "/content/Strata-data/", str(backup) + "/"], check=True)
print("モデルデータのバックアップ完了:", backup)

### 次回のセッションで復元するとき

Colabを再起動したら、**セットアップを実行する前に**Google Driveをマウントして以下を実行してください。GPUの種類・RAM・空きSSDが今回と同等か確認します。その後Strata本体を取得し、セットアップセルを再実行します。

In [ ]:
# 次回のセッションでのみ使用（通常は実行不要）
from google.colab import drive
import subprocess
from pathlib import Path

drive.mount("/content/drive")
source = Path("/content/drive/MyDrive/Strata-backup")
assert source.exists(), "Drive上にバックアップがありません"
Path("/content/Strata-data").mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-a", "--info=progress2", str(source) + "/", "/content/Strata-data/"], check=True)
print("復元完了。Strataのセットアップを続けてください")

---

**終わったら**：Colabのランタイムを接続解除・削除してCompute Unitsの消費を止めてください。

**スマホで開く場合**：Google ColabはWebサービスなのでアプリのインストールは不要です。Android/iPhoneのブラウザから開いて`.ipynb`をアップロードできます（メニューが見づらいときはPC版サイト表示を利用）。このノートブックはスマホへ保存できますが、**約68GBのモデルはスマホではなくColabの仮想マシンにダウンロードされます**。モデルの長時間ダウンロード・実行にはPCブラウザの方が操作しやすいです。